In [1]:
import pandas as pd
import numpy as np
import folium
from folium.plugins import MarkerCluster

# Gerando dados sintéticos de imóveis em Nova Iguaçu e Queimados
np.random.seed(42)
n_imoveis = 45

# Coordenadas base (Aproximadas)
# Nova Iguaçu: -22.756, -43.460
# Queimados: -22.716, -43.555

dados_imoveis = {
    'id_imovel': range(1, n_imoveis + 1),
    'cidade': np.where(np.random.rand(n_imoveis) > 0.4, 'Nova Iguaçu', 'Queimados'),
    'valor_venda': np.random.uniform(150000, 850000, n_imoveis).round(2),
    'tipo': np.random.choice(['Casa', 'Apartamento', 'Terreno'], n_imoveis)
}

df_mapa = pd.DataFrame(dados_imoveis)

# Atribuindo coordenadas com base na cidade adicionando uma pequena dispersão aleatória
def gerar_lat(cidade):
    if cidade == 'Nova Iguaçu':
        return -22.756 + np.random.uniform(-0.03, 0.03)
    return -22.716 + np.random.uniform(-0.02, 0.02)

def gerar_lon(cidade):
    if cidade == 'Nova Iguaçu':
        return -43.460 + np.random.uniform(-0.03, 0.03)
    return -43.555 + np.random.uniform(-0.02, 0.02)

df_mapa['latitude'] = df_mapa['cidade'].apply(gerar_lat)
df_mapa['longitude'] = df_mapa['cidade'].apply(gerar_lon)


# Parte 1: Inicialização e Marcadores Básicos


In [2]:
centro_lat = df_mapa['latitude'].mean()
centro_lon = df_mapa['longitude'].mean()

mapa = folium.Map(
    location=[centro_lat, centro_lon],
    zoom_start=12,
    tiles='OpenStreetMap'
)

In [ ]:
for _, imovel in df_mapa.head(5).iterrows():
    popup_texto = (
        f"Tipo: {imovel['tipo']}<br>"
        f"Valor de venda: R$ {imovel['valor_venda']:,.2f}"
    )

    folium.Marker(
        location=[imovel['latitude'], imovel['longitude']],
        popup=folium.Popup(popup_texto, max_width=300)
    ).add_to(mapa)

mapa

# Parte 2: Customização Visual com Marcadores Circulares

In [ ]:
centro_lat = df_mapa['latitude'].mean()
centro_lon = df_mapa['longitude'].mean()

mapa_circulos = folium.Map(
    location=[centro_lat, centro_lon],
    zoom_start=12,
    tiles='OpenStreetMap'
)

for _, imovel in df_mapa.iterrows():


    if imovel['cidade'] == 'Nova Iguaçu':
        cor = 'blue'
    else:
        cor = 'orange'

    folium.CircleMarker(
        location=[imovel['latitude'], imovel['longitude']],
        radius=8,
        color=cor,
        fill=True,
        fill_color=cor,
        fill_opacity=0.7,
        tooltip='Clique para detalhes'
    ).add_to(mapa_circulos)

mapa_circulos

# Parte 3: Agrupamento Inteligente (Clustering)

In [ ]:
centro_lat = df_mapa['latitude'].mean()
centro_lon = df_mapa['longitude'].mean()

mapa_cluster = folium.Map(
    location=[centro_lat, centro_lon],
    zoom_start=12,
    tiles='OpenStreetMap'
)

# Criando o objeto de agrupamento
cluster = MarkerCluster().add_to(mapa_cluster)

# Cores dos ícones conforme o tipo de imóvel
cores_tipo = {
    'Casa': 'green',
    'Apartamento': 'blue',
    'Terreno': 'gray'
}

# Adicionando todos os imóveis ao cluster
for _, imovel in df_mapa.iterrows():

    cor = cores_tipo[imovel['tipo']]

    popup_texto = (
        f"<b>Imóvel:</b> {imovel['id_imovel']}<br>"
        f"<b>Cidade:</b> {imovel['cidade']}<br>"
        f"<b>Tipo:</b> {imovel['tipo']}<br>"
        f"<b>Valor:</b> R$ {imovel['valor_venda']:,.2f}"
    )

    folium.Marker(
        location=[imovel['latitude'], imovel['longitude']],
        popup=folium.Popup(popup_texto, max_width=300),
        tooltip='Clique para detalhes',
        icon=folium.Icon(
            color=cor,
            icon='home',
            prefix='fa'
        )
    ).add_to(cluster)

# Salvando o mapa em arquivo HTML
mapa_cluster.save('mapa_imoveis_baixada.html')

# Exibindo o mapa
mapa_cluster